# SatQuery AI — ResNet-18 Land-Cover Fine-Tuning on BigEarthNet-S2
**19-Class CORINE-Derived Multi-Label Classification**

This notebook fine-tunes a torchvision ResNet-18 (ImageNet-pretrained) on a BigEarthNet-S2 subset using the 19-class label scheme.

**Target hardware:** Colab free T4 or Kaggle P100 — training should complete in 1–3 hours.

**Output:** Checkpoint saved to `models/land_cover/best_model.pt`, compatible with `models/download_models.py` and `satquery.classifiers.predict`.

In [ ]:
# Cell 1 — Install dependencies
!pip install -q torch torchvision torchgeo numpy Pillow huggingface_hub

In [ ]:
# Cell 2 — Configuration
import os

# Data
BIGEARTHNET_ROOT = "data/raw/bigearthnet"       # torchgeo archive location
SUBSET_DIR = "data/processed/bigearthnet_subset" # materialised .npz cache
N_PATCHES = 3000                                  # number of samples to prepare
BANDS = "s2"                                      # optical only for ResNet-18

# Training
EPOCHS = 10
BATCH_SIZE = 32
LR = 1e-3
FREEZE_BACKBONE_EPOCHS = 2

# Output
OUTPUT_DIR = "models/land_cover"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"Config: {N_PATCHES} patches, {EPOCHS} epochs, batch={BATCH_SIZE}, lr={LR}")

In [ ]:
# Cell 3 — Download BigEarthNet subset
# Uses torchgeo to download the official archive and materialise N_PATCHES
# into lightweight .npz files.

import subprocess, sys
from pathlib import Path

subset_path = Path(SUBSET_DIR)
if not list(subset_path.glob("sample_*.npz")):
    print("Downloading BigEarthNet subset via torchgeo …")
    print("(This fetches the full archive on first run — may take 15-30 min)")
    
    # Check if the download script exists (works when repo is cloned)
    script = Path("data/scripts/download_bigearthnet_subset.py")
    if script.exists():
        !python {str(script)} --bands {BANDS} --n-patches {N_PATCHES} \
            --root {BIGEARTHNET_ROOT} --out-dir {SUBSET_DIR}
    else:
        # Inline fallback if running outside the repo tree
        from torchgeo.datasets import BigEarthNet
        import numpy as np
        
        subset_path.mkdir(parents=True, exist_ok=True)
        ds = BigEarthNet(root=BIGEARTHNET_ROOT, split="train", bands=BANDS,
                         num_classes=19, download=True)
        n = min(N_PATCHES, len(ds))
        for i in range(n):
            sample = ds[i]
            np.savez_compressed(
                subset_path / f"sample_{i:06d}.npz",
                image=sample["image"].numpy(),
                label_mask=sample["label"].numpy().astype(bool),
            )
            if (i + 1) % 500 == 0:
                print(f"  {i+1}/{n} saved")
        print(f"Done: {n} samples saved to {SUBSET_DIR}")
else:
    n_existing = len(list(subset_path.glob("sample_*.npz")))
    print(f"Found {n_existing} existing samples in {SUBSET_DIR}. Skipping download.")

In [ ]:
# Cell 4 — Build model and dataset
import torch
import torch.nn as nn
import torchvision.models as models
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader, random_split
import numpy as np
from PIL import Image

BIGEARTHNET_19_CLASSES = [
    "Urban fabric", "Industrial or commercial units", "Arable land",
    "Permanent crops", "Pastures", "Complex cultivation patterns",
    "Land principally occupied by agriculture", "Broad-leaved forest",
    "Coniferous forest", "Mixed forest", "Natural grassland and sparsely vegetated areas",
    "Moors, heathland and sclerophyllous vegetation", "Sclerophyllous vegetation",
    "Transitional woodland, shrub", "Beaches, dunes, sands",
    "Inland wetlands", "Coastal wetlands", "Inland waters", "Marine waters",
]

# --- Model ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
backbone = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
features = nn.Sequential(*list(backbone.children())[:-1])
model = nn.Sequential(features, nn.Flatten(), nn.Linear(512, 19)).to(device)
print(f"ResNet-18 multi-label head initialised on {device}.")

# --- Transforms ---
train_tfm = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])
val_tfm = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# --- Dataset ---
class BENSubset(Dataset):
    def __init__(self, data_dir, transform=None):
        self.files = sorted(Path(data_dir).glob("sample_*.npz"))
        self.transform = transform or val_tfm
    def __len__(self):
        return len(self.files)
    def __getitem__(self, idx):
        d = np.load(self.files[idx])
        img = d["image"]
        lbl = d["label_mask"].astype(np.float32)
        # Extract RGB bands
        if img.shape[0] >= 4:
            rgb = img[[3,2,1], :, :]
        elif img.shape[0] >= 3:
            rgb = img[:3, :, :]
        else:
            rgb = np.stack([img[0]]*3, axis=0)
        rgb = rgb.astype(np.float32)
        if rgb.max() > 1.0:
            rgb = rgb / (rgb.max() + 1e-8)
        rgb_hwc = np.transpose(rgb, (1, 2, 0))
        pil = Image.fromarray((rgb_hwc * 255).clip(0,255).astype(np.uint8))
        return self.transform(pil), torch.from_numpy(lbl)

full_ds = BENSubset(SUBSET_DIR, train_tfm)
n_val = max(1, int(len(full_ds) * 0.2))
n_train = len(full_ds) - n_val
train_ds, val_ds = random_split(full_ds, [n_train, n_val])

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f"Train: {n_train}  Val: {n_val}  Classes: 19")

In [ ]:
# Cell 5 — Training loop
import time, json

criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

best_f1 = 0.0
log = []

for epoch in range(1, EPOCHS + 1):
    t0 = time.time()
    
    # Freeze backbone for first N epochs
    for p in model[0].parameters():
        p.requires_grad = (epoch > FREEZE_BACKBONE_EPOCHS)
    
    # Train
    model.train()
    train_loss = 0.0
    for imgs, lbls in train_loader:
        imgs, lbls = imgs.to(device), lbls.to(device)
        optimizer.zero_grad()
        loss = criterion(model(imgs), lbls)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * imgs.size(0)
    train_loss /= n_train
    scheduler.step()
    
    # Validate
    model.eval()
    val_loss = 0.0
    all_p, all_l = [], []
    with torch.no_grad():
        for imgs, lbls in val_loader:
            imgs, lbls = imgs.to(device), lbls.to(device)
            logits = model(imgs)
            val_loss += criterion(logits, lbls).item() * imgs.size(0)
            all_p.append((torch.sigmoid(logits) >= 0.5).cpu().numpy())
            all_l.append(lbls.cpu().numpy())
    val_loss /= n_val
    
    preds = np.concatenate(all_p)
    labels = np.concatenate(all_l)
    eps = 1e-8
    tp = (preds * labels).sum(axis=1)
    fp = (preds * (1 - labels)).sum(axis=1)
    fn = ((1 - preds) * labels).sum(axis=1)
    precision = tp / (tp + fp + eps)
    recall = tp / (tp + fn + eps)
    f1 = float((2 * precision * recall / (precision + recall + eps)).mean())
    
    elapsed = time.time() - t0
    log.append({"epoch": epoch, "train_loss": round(train_loss, 5),
                "val_loss": round(val_loss, 5), "val_f1": round(f1, 4),
                "time_s": round(elapsed, 1)})
    print(f"Epoch {epoch}/{EPOCHS}  train={train_loss:.4f}  val={val_loss:.4f}  F1={f1:.4f}  ({elapsed:.0f}s)")
    
    if f1 > best_f1:
        best_f1 = f1
        torch.save({"model_state_dict": model.state_dict(), "epoch": epoch, "val_f1": f1},
                   f"{OUTPUT_DIR}/best_model.pt")
        print(f"  ↑ Best model saved (F1={f1:.4f})")

print(f"\nTraining complete. Best F1: {best_f1:.4f}")
with open(f"{OUTPUT_DIR}/training_log.json", "w") as f:
    json.dump(log, f, indent=2)

In [ ]:
# Cell 6 — Validation metrics summary
import json
from pathlib import Path

with open(f"{OUTPUT_DIR}/training_log.json") as f:
    log = json.load(f)

print("\n" + "=" * 50)
print("  RESNET-18 FINE-TUNING SUMMARY")
print("=" * 50)
print(f"  Dataset:     BigEarthNet-S2 ({N_PATCHES} patches)")
print(f"  Classes:     19 (CORINE-derived)")
print(f"  Epochs:      {EPOCHS}")
print(f"  Best val F1: {best_f1:.4f}")
print(f"  Checkpoint:  {OUTPUT_DIR}/best_model.pt")
print(f"\n  Epoch | Train Loss | Val Loss | Val F1")
print(f"  {'─'*45}")
for e in log:
    print(f"  {e['epoch']:5d} | {e['train_loss']:10.5f} | {e['val_loss']:8.5f} | {e['val_f1']:.4f}")
print("=" * 50)

In [ ]:
# Cell 7 — (Optional) Push checkpoint to Hugging Face Hub
# Uncomment and fill in your HF token + repo name.

# from huggingface_hub import HfApi
# api = HfApi()
# api.upload_file(
#     path_or_fileobj=f"{OUTPUT_DIR}/best_model.pt",
#     path_in_repo="best_model.pt",
#     repo_id="your-username/satquery-land-cover-resnet18",
#     repo_type="model",
#     token="hf_YOUR_TOKEN_HERE",
# )
# print("Checkpoint pushed to HF Hub.")

print("Skipped HF Hub push — uncomment and configure above when ready.")
print(f"Checkpoint at: {OUTPUT_DIR}/best_model.pt")
print(f"models/download_models.py can serve it from 'satquery-ai/land-cover-resnet18-bigearthnet'.")